# CSRA old-100 development: scoring, analysis and sample-size scenarios

Use an **A100 GPU** for section 3. The exact old items and completed hardware profiles are included. This notebook accepts no fresh dataset and never generates Mistral development target outcomes. Its execution and analysis logic has been tested locally with artificial scores; the new GPU backend has not run locally. All outputs remain provisional while factual re-review is pending.

The reference scoring estimate is **2.9 A100 GPU-hours**, plus setup, token auditing, loading, interruptions and retries. CPU encoding, nested fitting and calibration are extra. Completed batches/folds/scenarios are reused. Keep the same Drive run folder when resuming, and never open that folder from two active runtimes.


## 1. Mount Drive and upload CSRA_Development.zip

Upload the new **CSRA_Development.zip** only if prompted. The notebook reuses its frozen package from Drive on later sessions. Keep earlier pilot and hardware-profile folders unchanged.


In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import zipfile, io, hashlib, stat

drive.mount('/content/drive')
RUN_DIR=Path('/content/drive/MyDrive/CSRA_Development_runs/run1')
FROZEN=RUN_DIR/'frozen_package'
if FROZEN.exists():
    ROOT=FROZEN
    print('Resuming the frozen development package:', ROOT)
else:
    ROOT=Path('/content/CSRA_Development')
    if not ROOT.exists():
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise RuntimeError('Upload CSRA_Development.zip only.')
        payload=next(iter(uploaded.values()))
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                p=PurePosixPath(member.filename)
                if p.is_absolute() or '..' in p.parts or not p.parts or p.parts[0]!=ROOT.name or stat.S_ISLNK(member.external_attr>>16):
                    raise RuntimeError('Unexpected archive path.')
            archive.extractall('/content')
for line in (ROOT/'SHA256SUMS.txt').read_text().splitlines():
    digest,name=line.split('  ',1)
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=digest:
        raise RuntimeError('Changed package file: '+name)
print('Package verified. Run folder:', RUN_DIR)


## 2. Install the pinned environment and initialize the run

The unused vision/audio CUDA extensions are removed. Installations and study scripts use fresh subprocesses, so no kernel restart is needed. Errors stream below. On a CPU session the same packages support sections 4–5, but section 3 needs CUDA. Do not run unrelated package-install cells afterward.


In [ ]:
import subprocess,sys,os

def visible(command):
    environment={**os.environ,'OMP_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1','TOKENIZERS_PARALLELISM':'false'}
    process=subprocess.Popen(list(map(str,command)),cwd=ROOT,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
        status=process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if status:
        raise RuntimeError(f'Command failed with exit {status}; see the traceback above. Resume this same Drive run after resolving the issue.')

def script(name,*args):
    visible([sys.executable,'-u',ROOT/'code'/name,*args])

visible([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'])
visible([sys.executable,'-m','pip','install','torch==2.11.0+cu128','--index-url','https://download.pytorch.org/whl/cu128'])
visible([sys.executable,'-m','pip','install','-r',ROOT/'requirements-scoring.txt','-r',ROOT/'requirements-analysis.txt'])
visible([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'])
script('run_development.py','validate')
script('run_development.py','prepare','--run-dir',RUN_DIR)
ROOT=FROZEN
print('Running the frozen package:',ROOT)


## 3. Run development scoring on the A100

This is the new old-100 development experiment. First audit **every possible input** and label continuation without using new outcomes. Then score the three known targets, derive their FP32 influence targets and recorded answers, and query all four reporters under both ownership cues. Each model runs in its own process to release GPU memory. Existing completed jobs are skipped before loading weights.

If interrupted or disconnected, rerun sections 1–2 and then this section. An interrupted final uncommitted line is preserved and recovered automatically; committed scores are never overwritten. A failure stops the pipeline at the actual traceback. Do not change item text, model revisions, precision, labels or prompts in place. No repeated full hardware qualification is required.


In [ ]:
script('run_development.py','audit','--run-dir',RUN_DIR)
for model in ('granite','phi','qwen'):
    print('Development answer stage:',model,flush=True)
    script('run_development.py','answers','--model',model,'--run-dir',RUN_DIR)
script('run_development.py','targets','--run-dir',RUN_DIR)
for model in ('granite','phi','qwen','mistral'):
    print('Development reporter stage:',model,flush=True)
    script('run_development.py','reports','--model',model,'--run-dir',RUN_DIR)
script('run_development.py','assemble','--run-dir',RUN_DIR)
script('run_development.py','status','--run-dir',RUN_DIR)
print('GPU scoring complete. All subsequent work uses CPU.')


## 4. CPU analysis and precision calibration

You may switch to a CPU runtime now to avoid consuming GPU time for this section. After switching, rerun sections 1–2 to restore Drive/package variables, **skip section 3**, and run this section. Staying in the current runtime also works; these scripts explicitly use CPU.

The encoder is pinned and never fine-tuned. Nested group-held-out fits preserve all shared subjects/triples. Calibration uses 1,000 artificial future samples and 1,999 bootstrap draws per base metric/N; results are planning scenarios, not observed fresh effects. Only four original-language answer-triple groups are available, and Mistral's target variance remains unobserved. The output cannot silently authorize fresh collection or certify power. Completed analysis folds and N/metric combinations are reused on resumption.

The primary Ridge analysis runs by default. The fixed secondary GradientBoosting learner is available in the package but is not used to choose a favorable primary learner. Factual review is still pending; the saved training artifact is explicitly marked as a draft.


In [ ]:
script('embed_inputs.py','--run-dir',RUN_DIR)
script('analyze_development.py','--run-dir',RUN_DIR)
script('calibrate_size.py','--run-dir',RUN_DIR)
print('Development analysis and precision scenarios saved to Drive.')


## 5. Download the results ZIP

Return **CSRA_Development_results.zip**. This includes committed scores, the frozen package, encoder/analysis manifests, a draft training artifact and the precision report. It contains no model weights, credentials or fresh-set results. This export can also be run after an interrupted stage to return diagnostics. The separate review/timing forms remain blank until real reviewers complete them; do not invent decisions or identities.


In [ ]:
RESULTS=Path('/content/CSRA_Development_results.zip')
script('run_development.py','export','--run-dir',RUN_DIR,'--output',RESULTS)
files.download(str(RESULTS))
